# Select sample size for surface ozone TMREL, beta and BMR

Selecting the number of samples is a balance between computational cost, given the high spatial resolution and associated memory requirements, and the need for a sufficiently large sample size to robustly span the uncertainty space.

Users can test their optimum sample size using `Test_sample_size.ipynb`. The example we use here is for 300 samples.

In [ ]:
import os
import xarray as xr
import numpy as np
from utils.utils import create_global_country_map
import config
from utils.utils import require_dir
import pathlib

In [ ]:
# === CHOOSE NUMBER OF SAMPLES ===
n_samples = 300

# Set the seed to produce reproducible random numbers
np.random.seed(42)

In [ ]:
# === Calculate the TMREL distribution ===

# TMREL from GBD21 (uniform distribution)
tmrel_low = 29.1
tmrel_high = 35.7
tmrel_samples = np.random.uniform(tmrel_low, tmrel_high, size=n_samples)

tmrel_da = xr.DataArray(
    tmrel_samples,
    dims=["samples"],
    coords={"samples": np.arange(n_samples)}
).astype("float32")

# === Save file in scratch directory ===
SAVE_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "TMREL")

out_file = f"TMREL_{n_samples}_samples_ozone.nc"
out_path = os.path.join(SAVE_DIR, out_file)
tmrel_da.to_netcdf(out_path)

In [ ]:
# === Calculate the beta distribution ===

# Beta from RR per 10ppb (normal distribution)
RR_10 = 1.074
RR_10_lower = 1.014
RR_10_upper = 1.137
beta_mean = np.log(RR_10) / 10
beta_std = (np.log(RR_10_upper) - np.log(RR_10_lower)) / (2 * 1.96 * 10)
beta_samples = np.random.normal(beta_mean, beta_std, size=n_samples)

beta_da = xr.DataArray(
    beta_samples,
    dims=["samples"],
    coords={"samples": np.arange(n_samples)}
).astype("float32")

# === Save file in scratch directory ===
SAVE_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "beta_ozone")

out_file = f"beta_{n_samples}_samples_ozone.nc"
out_path = os.path.join(SAVE_DIR, out_file)
beta_da.to_netcdf(out_path)

In [ ]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")

# Load BMR for each country (lat, lon, quantile)
bmr_file = "GBD_BMR_Country_COPD_newlabels_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)

MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")

In [ ]:
# === Calculate the BMR distribution ===

# BMR from VizHub (normal distribution)
bmr_mean = BMR.sel(quantile="mean")
bmr_lower = BMR.sel(quantile="lower")
bmr_upper = BMR.sel(quantile="upper")
bmr_std = (bmr_upper - bmr_lower) / (2 * 1.96)

bmr_samples = np.random.normal(
    bmr_mean,
    bmr_std,
    size=(n_samples, len(BMR.country)))

bmr_da = xr.DataArray(
    bmr_samples,
    dims=["samples", "country"],
    coords={"samples": np.arange(n_samples), "country": BMR.country}
).astype("float32")

del bmr_samples

# WARNING: this step can be slow and use a lot of memory
# e.g. ~100GB for 1000 samples, ~30GB for 200 samples
bmr_global = create_global_country_map(bmr_da, MASKS_DIR).chunk({"samples": 10, "lat": 180, "lon": 360})

In [ ]:
# === Save file in scratch directory ~25GB ===
SAVE_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "BMR_ozone")

out_file = f"GBD_BMR_Country_Mask_COPD_{n_samples}_samples_1990-2009.nc"
out_path = os.path.join(SAVE_DIR, out_file)
bmr_global.to_netcdf(out_path)